In [ ]:
"""
==========================================================================
BANK TRANSACTION FRAUD DETECTION - END TO END MACHINE LEARNING PIPELINE
==========================================================================

PROBLEM STATEMENT
------------------
Banks process millions of transactions daily, and only a very small
fraction of them are fraudulent (in this dataset, ~12.5%). The goal of
this project is to build a machine learning classifier that can flag a
transaction as FRAUD or NOT FRAUD in real time, using behavioural and
device/session-level signals (login attempts, device risk score,
geo-distance, transaction velocity, authentication type, etc.).

This is a classic **imbalanced binary classification** problem, and the
business objective is NOT plain accuracy -- it is:
    - Catch as many frauds as possible (high RECALL on fraud class)
    - Without drowning genuine customers in false alarms (reasonable PRECISION)
So the project is evaluated primarily using Precision, Recall, F1-score,
ROC-AUC and PR-AUC, not accuracy alone.

PIPELINE STEPS
--------------
1. Load & explore data (EDA)
2. Clean & preprocess (encode categoricals, scale numeric features)
3. Train/test split (stratified, to preserve fraud ratio)
4. Handle class imbalance using SMOTE (oversampling minority class)
5. Train multiple models: Logistic Regression, Random Forest, XGBoost
6. Evaluate with Accuracy, Precision, Recall, F1, ROC-AUC, Confusion Matrix
7. Pick the best model, plot feature importance & ROC curve
8. Save the trained model + preprocessing pipeline with joblib

Run:
    python fraud_detection.py
==========================================================================
"""


In [2]:

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import joblib

RANDOM_STATE = 42
DATA_PATH = "banking_transactions.csv"




In [3]:
# --------------------------------------------------------------------
# 1. LOAD DATA
# --------------------------------------------------------------------
def load_data(path=DATA_PATH):
    df = pd.read_csv(path)
    print(f"Dataset shape: {df.shape}")
    print(f"\nFraud distribution:\n{df['fraud_flag'].value_counts()}")
    fraud_pct = df["fraud_flag"].mean() * 100
    print(f"Fraud rate: {fraud_pct:.2f}%")
    return df

In [4]:
# --------------------------------------------------------------------
# 2. EXPLORATORY DATA ANALYSIS (saves plots to outputs/)
# --------------------------------------------------------------------
def run_eda(df, out_dir="outputs"):
    import os
    os.makedirs(out_dir, exist_ok=True)

    # Class balance plot
    plt.figure(figsize=(5, 4))
    df["fraud_flag"].value_counts().plot(kind="bar", color=["#4C72B0", "#DD8452"])
    plt.title("Fraud vs Non-Fraud Transaction Count")
    plt.xlabel("Fraud Flag")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(f"{out_dir}/class_balance.png", dpi=120)
    plt.close()

    # Correlation heatmap for numeric features
    numeric_df = df.select_dtypes(include=[np.number]).drop(columns=["transaction_id"])
    plt.figure(figsize=(11, 9))
    sns.heatmap(numeric_df.corr(), cmap="coolwarm", center=0, annot=False)
    plt.title("Feature Correlation Heatmap")
    plt.tight_layout()
    plt.savefig(f"{out_dir}/correlation_heatmap.png", dpi=120)
    plt.close()

    print(f"EDA plots saved to '{out_dir}/'")


In [5]:
# --------------------------------------------------------------------
# 3. PREPROCESSING
# --------------------------------------------------------------------
def build_preprocessor(numeric_features, categorical_features):
    numeric_transformer = StandardScaler()
    categorical_transformer = OneHotEncoder(handle_unknown="ignore", drop="first")

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )
    return preprocessor

In [6]:
# --------------------------------------------------------------------
# 4. TRAIN / EVALUATE MODELS
# --------------------------------------------------------------------
def evaluate_model(name, model, X_test, y_test, results):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    print(f"\n--- {name} ---")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"ROC-AUC  : {auc:.4f}")
    print(classification_report(y_test, y_pred, target_names=["Not Fraud", "Fraud"]))

    results[name] = {
        "accuracy": acc, "precision": prec, "recall": rec,
        "f1": f1, "roc_auc": auc, "y_pred": y_pred, "y_proba": y_proba
    }
    return results


def plot_confusion_matrix(y_test, y_pred, name, out_dir="outputs"):
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(4.5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Not Fraud", "Fraud"],
                yticklabels=["Not Fraud", "Fraud"])
    plt.title(f"Confusion Matrix - {name}")
    plt.ylabel("Actual")
    plt.xlabel("Predicted")
    plt.tight_layout()
    plt.savefig(f"{out_dir}/confusion_matrix_{name.replace(' ', '_')}.png", dpi=120)
    plt.close()


def plot_roc_curves(results, y_test, out_dir="outputs"):
    plt.figure(figsize=(6, 5))
    for name, res in results.items():
        fpr, tpr, _ = roc_curve(y_test, res["y_proba"])
        plt.plot(fpr, tpr, label=f"{name} (AUC = {res['roc_auc']:.3f})")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve Comparison")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{out_dir}/roc_curve_comparison.png", dpi=120)
    plt.close()


def plot_feature_importance(model, feature_names, out_dir="outputs"):
    importances = model.feature_importances_
    idx = np.argsort(importances)[-15:]
    plt.figure(figsize=(8, 6))
    plt.barh(range(len(idx)), importances[idx], color="#4C72B0")
    plt.yticks(range(len(idx)), [feature_names[i] for i in idx])
    plt.xlabel("Importance")
    plt.title("Top 15 Feature Importances (Best Model)")
    plt.tight_layout()
    plt.savefig(f"{out_dir}/feature_importance.png", dpi=120)
    plt.close()



In [7]:
# --------------------------------------------------------------------
# MAIN
# --------------------------------------------------------------------
def main():
    df = load_data()
    run_eda(df)

    # Features / target
    target = "fraud_flag"
    drop_cols = ["transaction_id"]
    X = df.drop(columns=[target] + drop_cols)
    y = df[target].astype(int)

    numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = X.select_dtypes(include=["object", "str"]).columns.tolist()
    print(f"\nNumeric features ({len(numeric_features)}): {numeric_features}")
    print(f"Categorical features ({len(categorical_features)}): {categorical_features}")

    # Train/test split (stratified to preserve the 12.5% fraud ratio)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )

    # Preprocess
    preprocessor = build_preprocessor(numeric_features, categorical_features)
    X_train_proc = preprocessor.fit_transform(X_train)
    X_test_proc = preprocessor.transform(X_test)

    feature_names = (
        numeric_features
        + list(preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_features))
    )

    # Handle class imbalance with SMOTE (only on training data)
    print(f"\nBefore SMOTE: {np.bincount(y_train)}")
    smote = SMOTE(random_state=RANDOM_STATE)
    X_train_bal, y_train_bal = smote.fit_resample(X_train_proc, y_train)
    print(f"After SMOTE : {np.bincount(y_train_bal)}")

    results = {}

    # --- Model 1: Logistic Regression (baseline) ---
    log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    log_reg.fit(X_train_bal, y_train_bal)
    results = evaluate_model("Logistic Regression", log_reg, X_test_proc, y_test, results)
    plot_confusion_matrix(y_test, results["Logistic Regression"]["y_pred"], "Logistic Regression")

    # --- Model 2: Random Forest ---
    rf = RandomForestClassifier(
        n_estimators=300, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1
    )
    rf.fit(X_train_bal, y_train_bal)
    results = evaluate_model("Random Forest", rf, X_test_proc, y_test, results)
    plot_confusion_matrix(y_test, results["Random Forest"]["y_pred"], "Random Forest")

    # --- Model 3: XGBoost ---
    xgb = XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1
    )
    xgb.fit(X_train_bal, y_train_bal)
    results = evaluate_model("XGBoost", xgb, X_test_proc, y_test, results)
    plot_confusion_matrix(y_test, results["XGBoost"]["y_pred"], "XGBoost")

    # --- Compare & pick best model (by F1-score on fraud class) ---
    plot_roc_curves(results, y_test)

    best_name = max(results, key=lambda k: results[k]["f1"])
    print(f"\n{'='*60}")
    print(f"BEST MODEL: {best_name} (F1 = {results[best_name]['f1']:.4f}, "
          f"ROC-AUC = {results[best_name]['roc_auc']:.4f})")
    print(f"{'='*60}")

    best_model = {"Logistic Regression": log_reg, "Random Forest": rf, "XGBoost": xgb}[best_name]

    if hasattr(best_model, "feature_importances_"):
        plot_feature_importance(best_model, feature_names)

    # Summary table
    summary = pd.DataFrame(results).T[["accuracy", "precision", "recall", "f1", "roc_auc"]]
    summary = summary.astype(float).round(4)
    print("\nModel comparison summary:\n", summary)
    summary.to_csv("outputs/model_comparison.csv")

    # Save pipeline (preprocessor + best model) for deployment/inference
    joblib.dump(preprocessor, "outputs/preprocessor.joblib")
    joblib.dump(best_model, "outputs/best_model.joblib")
   


if __name__ == "__main__":
    main()

Dataset shape: (10000, 20)

Fraud distribution:
fraud_flag
False    8749
True     1251
Name: count, dtype: int64
Fraud rate: 12.51%
EDA plots saved to 'outputs/'

Numeric features (16): ['transaction_amount', 'login_attempts', 'device_risk_score', 'transfer_frequency', 'anomaly_score', 'account_age_days', 'transaction_time_hour', 'failed_transactions_last_30d', 'avg_monthly_balance', 'daily_transaction_count', 'geo_distance_km', 'session_duration_minutes', 'transaction_velocity_score', 'card_present_flag', 'international_transaction_flag', 'suspicious_ip_flag']
Categorical features (2): ['payment_channel', 'authentication_type']

Before SMOTE: [6999 1001]
After SMOTE : [6999 6999]

--- Logistic Regression ---
Accuracy : 0.9320
Precision: 0.6638
Recall   : 0.9240
F1-score : 0.7726
ROC-AUC  : 0.9787
              precision    recall  f1-score   support

   Not Fraud       0.99      0.93      0.96      1750
       Fraud       0.66      0.92      0.77       250

    accuracy               